# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR² dataset using the `mlcroissant` library and follows a step-by-step approach to examine its structure and content.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets and fields by their @id
print("Record sets in the dataset:")
for rs in metadata.record_sets:
    print(f"  RecordSet: {rs['@id']} (name: {rs.get('name','')})")
    print("    Fields:")
    for field in rs['fields']:
        field_name = field.get('name', '')
        print(f"      Field: {field['@id']} (name: {field_name})")
        if 'columns' in field and field['columns']:
            for col in field['columns']:
                print(f"        Column: {col['@id']} (name: {col.get('name','')}, dataType: {col.get('dataType','')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Prepare a list of all record set IDs
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    # Load all records for each record set by @id
    print(f"Loading records for {record_set_id}...")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"Columns: {list(dataframes[record_set_id].columns)}\nSample: ")
        display(dataframes[record_set_id].head())
    else:
        print(f"No records found for {record_set_id}\n")
# For further EDA, select the first (main) record set
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Main record set selected for analysis: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For the demonstration, select a numeric field from the main record set
import numpy as np
main_df = dataframes.get(main_record_set_id)
if main_df is not None and not main_df.empty:
    print(f"Available columns: {main_df.columns.tolist()}")
    # Heuristically pick a numeric column (try common names)
    candidate_numeric_fields = [col for col in main_df.columns if any(keyword in col.lower() for keyword in ["count", "abundance", "weight", "coverage", "mw", "intensity"])]
    if not candidate_numeric_fields:
        # Fallback: try float columns
        numeric_fields = main_df.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_fields:
            numeric_field = numeric_fields[0]
        else:
            raise Exception("No obvious numeric field found for EDA.")
    else:
        numeric_field = candidate_numeric_fields[0]
    print(f"Selected numeric field '{numeric_field}' for filtering and normalization.")
    
    threshold = main_df[numeric_field].mean() # Use mean as threshold
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold:.3f} (n={len(filtered_df)}):")
    display(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
    print(f"Normalized '{numeric_field}' for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Grouping by a categorical field if one exists
    candidate_group_fields = [col for col in main_df.columns if any(key in col.lower() for key in ["class", "type", "category", "protein", "accession", "sample"])]
    group_field = candidate_group_fields[0] if candidate_group_fields else None
    if group_field and group_field in main_df.columns:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by '{group_field}':")
        display(grouped_df.head())
else:
    print("No data available in the main record set for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_df is not None and not main_df.empty and numeric_field in main_df.columns:
    plt.figure(figsize=(8,5))
    sns.histplot(main_df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Frequency")
    plt.show()
    
    # If a group field was selected previously, show boxplot
    if group_field and group_field in main_df.columns:
        plt.figure(figsize=(12,6))
        sns.boxplot(x=main_df[group_field], y=main_df[numeric_field])
        plt.title(f"{numeric_field} by {group_field}")
        plt.xticks(rotation=45)
        plt.show()


## 6. Conclusion
This notebook provided an overview and exploratory analysis of the FAIR² mass spectrometry dataset using the Croissant schema via the `mlcroissant` library. We loaded the data, reviewed its structure, performed basic filtering and normalization on numeric fields, visualized distributions, and highlighted how to access data by `@id`. Adjust code cells to target specific record sets or fields (using their `@id`) for deeper analysis as needed.